# Epinet

In [ ]:
%%capture
%pip install git+https://github.com/lightning-uq-box/lightning-uq-box.git

## Theoretic Foundation

An *epistemic neural network* (ENN), introduced by [Osband et al., 2023](https://arxiv.org/abs/2107.08924), is a pair: a function class $f_\theta(x, z)$ and a reference distribution $P_Z$ over an **epistemic index** $z$. Where a conventional network maps an input to a single prediction, an ENN maps an input *and an index* to a prediction, and integrating over $z \sim P_Z$ recovers a predictive distribution.

What this buys is *joint* predictions. Most UQ methods are evaluated one input at a time, on marginal predictions $\hat{P}(y_t \mid x_t)$. But many downstream uses — sequential decision making, active learning, bandits — depend on how predictions at *different* inputs covary, which the marginals cannot express. An ENN's joint prediction over $\tau$ inputs is

$$\hat{P}_{1:\tau}(y_{1:\tau}) = \int P_Z(dz) \prod_{t=1}^{\tau} \mathrm{softmax}(f_\theta(x_t, z))_{y_t},$$

where the shared $z$ inside the product is what ties the predictions together.

The **epinet** is a particular, cheap way of building one. Rather than a new architecture, it is a small head bolted onto a conventional base network $\mu_\zeta(x)$:

$$f_\theta(x, z) = \mu_\zeta(x) + \sigma_\eta(\mathrm{sg}[\phi_\zeta(x)], z),$$

where $\phi_\zeta(x)$ are features of the base network's last hidden layer and $\mathrm{sg}[\cdot]$ is a stop-gradient: the epinet reads the base network's features but never sends gradients back through them. The epinet itself splits into a learnable part and a *fixed* part,

$$\sigma_\eta(\tilde{x}, z) = \sigma^L_\eta(\tilde{x}, z) + \sigma^P(\tilde{x}, z), \qquad \sigma^L_\eta(\tilde{x}, z) := \mathrm{mlp}_\eta([\tilde{x}, z])^T z.$$

The learnable part is an MLP whose output is reshaped to $\mathbb{R}^{D_Z \times C}$ and contracted with the index. The fixed part $\sigma^P$ is a frozen, randomly initialized *prior function*. It is never trained, and it is what supplies uncertainty where the data does not pin the model down: away from the training data nothing pulls the learnable part toward cancelling it, so the prior's variation across $z$ survives as predictive spread.

Two properties make this practical. Because the base network is only read, never modified, an epinet can be attached to a **large pretrained network whose weights stay frozen** — the paper's headline result is an epinet beating a 100-particle deep ensemble on ImageNet joint log-loss at less than the cost of two particles. And because only the small head sees the index, averaging the loss over many index draws costs one base forward pass, not many.

## Imports

In [ ]:
import os
import tempfile
from functools import partial

import matplotlib.pyplot as plt
import torch
from lightning import Trainer
from lightning.pytorch import seed_everything
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.optim import Adam

from lightning_uq_box.datamodules import TwoMoonsDataModule
from lightning_uq_box.eval_utils import joint_log_loss_dyadic, marginal_log_loss
from lightning_uq_box.models import MLP
from lightning_uq_box.uq_methods import EpinetClassification
from lightning_uq_box.viz_utils import (
    plot_predictions_classification,
    plot_training_metrics,
    plot_two_moons_data,
)

plt.rcParams["figure.figsize"] = [14, 5]

In [ ]:
seed_everything(0)

We define a temporary directory to look at some training metrics and results.

In [ ]:
my_temp_dir = tempfile.mkdtemp()

## Datamodule

We use the Two Moons dataset, a two-dimensional binary classification problem. Its appeal here is that the input space is small enough to visualize completely, so we can see exactly where the model is and is not confident.

In [ ]:
dm = TwoMoonsDataModule(batch_size=128)

X_train, Y_train, X_test, Y_test, test_grid_points = (
    dm.X_train,
    dm.Y_train,
    dm.X_test,
    dm.Y_test,
    dm.test_grid_points,
)

In [ ]:
fig = plot_two_moons_data(X_train, Y_train, X_test, Y_test)

## Model

As in the regression case we begin with an ordinary base network, which the epinet will read from without modifying.

In [ ]:
network = MLP(n_inputs=2, n_hidden=[50, 50], n_outputs=2, activation_fn=nn.ReLU())
network

`EpinetClassification` wraps it. The loss is the ordinary cross-entropy: the epinet needs no special loss function, because averaging the standard loss over sampled indices is exactly what the paper prescribes.

In [ ]:
epinet_module = EpinetClassification(
    model=network,
    optimizer=partial(Adam, lr=1e-2),
    loss_fn=nn.CrossEntropyLoss(),
    task="multiclass",
    index_dim=8,
    num_index_samples=8,
    num_pred_samples=100,
    epinet_hidden_dims=[15, 15],
    prior_hidden_dims=[5, 5],
    epi_prior_scale=0.0,
    input_prior_scale=3.0,
)

## Trainer

In [ ]:
logger = CSVLogger(my_temp_dir)
trainer = Trainer(
    accelerator="cpu",
    max_epochs=100,  # number of epochs we want to train
    logger=logger,  # log training metrics for later evaluation
    log_every_n_steps=1,
    enable_checkpointing=False,
    enable_progress_bar=False,
    default_root_dir=my_temp_dir,
)

In [ ]:
trainer.fit(epinet_module, dm)

## Training Metrics

In [ ]:
fig = plot_training_metrics(
    os.path.join(my_temp_dir, "lightning_logs"), ["train_loss", "trainAcc"]
)

## Prediction

In [ ]:
# save predictions
trainer.test(epinet_module, dm.test_dataloader())

In [ ]:
preds = epinet_module.predict_step(test_grid_points.to(epinet_module.device))

## Evaluate Predictions

Evaluating over the full input grid shows where the epinet is uncertain. The uncertainty should be low along the two moons, where the training data pins the model down, and rise in the gap between them and away from the data entirely.

In [ ]:
fig = plot_predictions_classification(
    X_test, Y_test, preds["pred"].argmax(-1), test_grid_points, preds["pred_uct"]
)

## Joint Predictions

Everything so far has been a marginal evaluation: each point scored on its own. The point of an ENN, though, is the *joint* prediction, and the box ships the metrics to measure it.

`marginal_log_loss` collapses the index samples into one predictive distribution per input and scores each independently. `joint_log_loss_dyadic` keeps the samples tied across a batch of $\\tau$ inputs and scores the batch as a whole, drawing those batches by **dyadic sampling** ([Appendix F](https://arxiv.org/abs/2107.08924)): pick $\\kappa$ anchor points, then resample $\\tau$ points from just those anchors. The repetition is deliberate — a model that treats its own predictions as independent pays for the same mistake several times over, and a model that has captured the dependence does not.

Where we evaluate matters. On the test set the model is almost perfectly confident and correct, so both losses sit near zero and the two metrics have nothing to distinguish. The interesting region is the decision boundary, where the model is genuinely unsure — so we evaluate on the grid points with the highest predictive uncertainty, and use the epinet's own most-likely labels there as the targets.

These functions take logits of shape `[num_samples, num_data, num_classes]`, so we transpose the `[num_data, num_classes, num_samples]` layout that `predict_step` returns.

In [ ]:
def log_losses(logits, targets, seed=0):
    """Marginal and dyadic joint log loss for one set of ENN logits."""
    generator = torch.Generator().manual_seed(seed)
    marginal = marginal_log_loss(logits, targets)
    joint = joint_log_loss_dyadic(
        logits, targets, tau=10, kappa=2, num_batches=200, generator=generator
    )
    return float(marginal), float(joint)


# the 200 grid points the model is least sure about, i.e. the decision boundary
boundary = torch.topk(preds["pred_uct"], 200).indices
boundary_logits = preds["logits"][boundary].permute(2, 0, 1).cpu()
boundary_targets = preds["pred"][boundary].argmax(-1).cpu()

# the ordinary test set, for contrast
test_preds = epinet_module.predict_step(X_test.to(epinet_module.device))
test_logits = test_preds["logits"].permute(2, 0, 1).cpu()
test_targets = Y_test.long().cpu()

for name, lg, tg in [
    ("decision boundary", boundary_logits, boundary_targets),
    ("test set", test_logits, test_targets),
]:
    marginal, joint = log_losses(lg, tg)
    print(f"{name:>18}:  marginal {marginal:.4f}   joint (tau=10) {joint:.4f}")

On the test set both losses are essentially zero: the model is confident and right, and there is no dependence left to capture. Along the decision boundary the joint log loss comes out **below** the marginal one, which is the signature of a model whose uncertainty is correlated across inputs in a useful way. Its index samples disagree about where the boundary lies, but each individual sample is self-consistent — so repeatedly querying nearby points costs it far less than it would cost a predictor that treats each query as an independent coin flip.

That gap is what the joint metric exists to measure, and it is invisible to any marginal metric. Because every method in the box produces sampled predictions in this same layout, the same two functions can compare them all on the footing the paper argues actually matters.